# LoRA fine-tuning Qwen2.5-0.5B-Instruct to score interview answers (1–5)

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU* (free). Total ≈ 15–25 min.

What this notebook does
1. Loads the SAME `train / val / test` splits used by the scikit-learn and Keras baselines (`ml/data/`).
2. Scores the test set with the **base** model zero-shot.
3. Fine-tunes with **LoRA** (only ~1% of weights trained) on `train`, early-stopping on `val`.
4. Scores the test set again and compares base vs LoRA vs the classic baselines.
5. Exports the adapter, a merged model, and a **GGUF + Ollama Modelfile** to run it locally on CPU.

**Scoring method:** we never parse free text. After the prompt, we read the model's next-token
logits for the tokens `1`…`5` and take the most likely one (and the probability-weighted expected
score). That's deterministic, fast, and can't produce an invalid output.

In [ ]:
# Colab already has PyTorch. ~1 min.
!pip -q install -U "transformers>=4.45" "peft>=0.13" accelerate pandas scikit-learn

In [ ]:
import copy, json, math, os, random, time
import numpy as np, pandas as pd, torch

REPO_URL = "https://github.com/hatimsoni/mockai.git"   # change if you forked the repo
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 42
# SMOKE_TEST = tiny run on CPU to check the code works (used by run_local_smoke.py). Real run needs a GPU.
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1" or not torch.cuda.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
EPOCHS = 1 if SMOKE_TEST else 3
# Effective batch 16 either way. The logits tensor (batch x tokens x 151,936 vocab) dominates
# memory, so CPU smoke runs use micro-batches of 2 with more gradient accumulation.
BATCH_SIZE, GRAD_ACCUM = (2, 8) if SMOKE_TEST else (8, 2)
LR = 2e-4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"device={DEVICE} smoke_test={SMOKE_TEST}", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
# Data: clone the repo (public) or upload train.csv / val.csv / test.csv when prompted.
DATA_DIR = os.environ.get("DATA_DIR")
if DATA_DIR is None:
    if not os.path.exists("mockai"):
        os.system(f"git clone --depth 1 {REPO_URL}")
    if os.path.exists("mockai/ml/data/train.csv"):
        DATA_DIR = "mockai/ml/data"
    else:
        from google.colab import files
        files.upload()
        DATA_DIR = "."
train, val, test = (pd.read_csv(f"{DATA_DIR}/{s}.csv") for s in ("train", "val", "test"))
if SMOKE_TEST:
    train, val, test = train.sample(24, random_state=SEED), val.sample(8, random_state=SEED), test.sample(12, random_state=SEED)
print(len(train), len(val), len(test), "rows | test labels:", test.label.value_counts().sort_index().to_dict())

In [ ]:
# EXACT copy of ai/app/scoring_prompt.py (generated by build_notebook.py) — the served model
# must see the same prompt it was trained on.
SYSTEM = "You are an interview coach. Score the candidate's answer from 1 (very poor) to 5 (excellent). Reply with a single digit."

USER_TEMPLATE = """Rubric: 1 off-topic or non-answer; 2 on-topic but vague; 3 one concrete example but missing result or reasoning; 4 specific and structured with a measurable result; 5 clear STAR structure, decisions with trade-offs, own role, quantified impact.

Question: {question}
Answer: {answer}

Score (1-5):"""

def build_messages(question, answer):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": USER_TEMPLATE.format(question=question, answer=answer)}]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"          # left padding -> the LAST position is always "the next token"
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE).to(DEVICE)

# Qwen's tokenizer has one token per digit; we score by comparing these 5 logits.
DIGIT_IDS = [tok.encode(str(d), add_special_tokens=False) for d in range(1, 6)]
assert all(len(t) == 1 for t in DIGIT_IDS), DIGIT_IDS
DIGIT_IDS = [t[0] for t in DIGIT_IDS]
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

In [ ]:
def prompt_text(q, a):
    return tok.apply_chat_template(build_messages(q, a), tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score(model, df, batch_size=16):
    """Returns (predicted class 1-5, expected score) from next-token logits of '1'..'5'."""
    model.eval()
    prompts = [prompt_text(q, a) for q, a in zip(df.question, df.answer)]
    preds, expected = [], []
    for i in range(0, len(prompts), batch_size):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(DEVICE)
        logits = model(**enc).logits[:, -1, :]               # next-token distribution
        p = logits[:, DIGIT_IDS].float().softmax(-1).cpu().numpy()
        preds += (p.argmax(1) + 1).tolist()
        expected += (p @ np.arange(1, 6)).tolist()
    return np.array(preds), np.array(expected)

def evaluate(y_true, y_pred):
    """Same metrics as ml/classic/metrics.py."""
    from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, mean_absolute_error
    y_true = np.asarray(y_true, int); y_pred = np.clip(np.rint(y_pred), 1, 5).astype(int)
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 3),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0)), 3),
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 3),
        "within_1": round(float(np.mean(np.abs(y_true - y_pred) <= 1)), 3),
        "qwk": round(float(cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=[1, 2, 3, 4, 5])), 3),
        "n": int(len(y_true)),
    }

t0 = time.time()
base_pred, _ = score(model, test)
base_metrics = evaluate(test.label, base_pred)
print("BASE zero-shot:", base_metrics, f"({time.time() - t0:.0f}s)")

## LoRA in one paragraph
Full fine-tuning updates all 494M weights (needs lots of GPU memory). **LoRA** freezes them and adds,
next to each chosen weight matrix `W` (d×d), two small matrices `A` (d×r) and `B` (r×d) with r=16.
The layer computes `W·x + (alpha/r)·B·A·x`. Only A and B are trained (~1% of parameters), `B` starts
at zero so training starts exactly from the base model, and afterwards `B·A` can be **merged** into `W`
— zero extra inference cost.

In [ ]:
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)      # adapters are created in float32 for stable training
model.print_trainable_parameters()

In [ ]:
# Completion-only training: the loss is computed ONLY on the answer tokens ("4<|im_end|>"),
# not on the prompt (label -100 = ignored), so the model learns to score, not to copy prompts.
def encode(q, a, label):
    p_ids = tok(prompt_text(q, a), add_special_tokens=False)["input_ids"]
    t_ids = tok(f"{label}<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": p_ids + t_ids, "labels": [-100] * len(p_ids) + t_ids}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    ids = torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch])
    labels = torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch])
    mask = torch.tensor([[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch])
    return {"input_ids": ids.to(DEVICE), "labels": labels.to(DEVICE), "attention_mask": mask.to(DEVICE)}

train_ex = [encode(q, a, l) for q, a, l in zip(train.question, train.answer, train.label)]
val_ex = [encode(q, a, l) for q, a, l in zip(val.question, val.answer, val.label)]
print("max tokens:", max(len(e["input_ids"]) for e in train_ex))

In [ ]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train_ex) / (BATCH_SIZE * GRAD_ACCUM))
total = steps_per_epoch * EPOCHS
warmup = max(1, int(0.06 * total))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, (total - s) / max(1, total - warmup)))
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def val_loss():
    model.eval(); losses = []
    for i in range(0, len(val_ex), BATCH_SIZE):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            losses.append(model(**collate(val_ex[i:i + BATCH_SIZE])).loss.item())
    return float(np.mean(losses))

best, best_state, t0 = float("inf"), None, time.time()
for epoch in range(EPOCHS):
    model.train(); random.shuffle(train_ex)
    for step, i in enumerate(range(0, len(train_ex), BATCH_SIZE)):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            loss = model(**collate(train_ex[i:i + BATCH_SIZE])).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or i + BATCH_SIZE >= len(train_ex):
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
    vl = val_loss()
    print(f"epoch {epoch + 1}: val_loss={vl:.4f} ({(time.time() - t0) / 60:.1f} min)")
    if vl < best:                                   # keep the best adapter (early stopping)
        best, best_state = vl, copy.deepcopy(get_peft_model_state_dict(model))
set_peft_model_state_dict(model, best_state)
train_minutes = round((time.time() - t0) / 60, 1)

In [ ]:
lora_pred, lora_expected = score(model, test)
lora_metrics = evaluate(test.label, lora_pred)

rows = {"qwen_base_zero_shot": base_metrics, "qwen_lora": lora_metrics}
results_dir = os.path.join(DATA_DIR, "..", "results")
for f in ("sklearn_metrics.json", "keras_metrics.json"):         # classic baselines, same test set
    path = os.path.join(results_dir, f)
    if os.path.exists(path) and not SMOKE_TEST:
        rows.update(json.load(open(path))["results"])
table = pd.DataFrame(rows).T[["accuracy", "macro_f1", "mae", "within_1", "qwk"]]
print(table.to_string())

out = {"results": {"qwen_base_zero_shot": base_metrics, "qwen_lora": lora_metrics},
       "base_model": BASE_MODEL, "lora": {"r": 16, "alpha": 32, "epochs": EPOCHS},
       "train_minutes": train_minutes, "smoke_test": SMOKE_TEST}
json.dump(out, open("lora_metrics.json", "w"), indent=2)
print(json.dumps(out, indent=2))

In [ ]:
# Save the small adapter (~35 MB) and a merged full model (for GGUF / transformers).
model.save_pretrained("mockai-scorer-lora")
merged = model.merge_and_unload()            # W := W + (alpha/r)·B·A  -> plain Qwen model again
merged.save_pretrained("mockai-scorer-merged", safe_serialization=True)
tok.save_pretrained("mockai-scorer-merged")

In [ ]:
# Convert to GGUF (8-bit) for Ollama / llama.cpp on a CPU laptop, write a Modelfile, zip, download.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py mockai-scorer-merged --outtype q8_0 --outfile mockai-scorer-q8_0.gguf

modelfile = f'''FROM ./mockai-scorer-q8_0.gguf
TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
"""
SYSTEM """{SYSTEM}"""
PARAMETER temperature 0
PARAMETER num_predict 2
PARAMETER stop "<|im_end|>"
'''
open("Modelfile", "w").write(modelfile)
!zip -q -r mockai-scorer.zip mockai-scorer-q8_0.gguf Modelfile lora_metrics.json mockai-scorer-lora
from google.colab import files
files.download("mockai-scorer.zip")

## Use it locally
1. Unzip `mockai-scorer.zip` somewhere, then: `ollama create mockai-scorer -f Modelfile`
2. Copy `lora_metrics.json` to `ml/results/` and run `python ml/results/compare.py` to update the table.
3. Set `SCORER_MODEL=mockai-scorer` for the AI service: `/ml/quality` then also returns `finetuned_score`.